In [0]:
# ---- Silver config  ----
CATALOG = "dbr_dev"
SCHEMA  = "live_transit_monitor"

BRONZE   = f"{CATALOG}.{SCHEMA}.gps_data"
VEHICLES = f"{CATALOG}.{SCHEMA}.bronze_vehicles"
ROUTES   = f"{CATALOG}.{SCHEMA}.bronze_gtfs_routes"
SILVER   = f"{CATALOG}.{SCHEMA}.gps_positions_silver"

In [0]:
bronze = spark.read.table(BRONZE)
display(bronze.limit(10))   
bronze.selectExpr("min(event_time) min_t", "max(event_time) max_t",
              "count(distinct vehicleId) vehicles").show()

In [0]:
bronze.printSchema()

# lastUpdate, generate needs to be changed to timestamp

In [0]:
# Checking if there are any nulls in the table
from pyspark.sql.functions import col, count, when

bronze.select([count(when(col(c).isNull(), c)).alias(c) for c in bronze.columns]).show()

# Nulls when a vehicle is not on a trip (returning to car barn)

In [0]:
from pyspark.sql import functions as F, Window

#bronze = spark.read.table(BRONZE)

silver = (bronze
    .withColumn("generated", F.to_timestamp("generated"))
    .withColumn("lastUpdate", F.to_timestamp("lastUpdate"))
    .withColumn("scheduledTripStartTime", F.to_timestamp("scheduledTripStartTime"))
    .dropDuplicates(["vehicleId", "generated"])
    .withColumn("delay_min", F.round(F.col("delay") / 60.0, 1))
    .withColumn("has_trip", F.col("tripId").isNotNull())   # vehicle currently on a scheduled trip
    .withColumn("delay_bucket",
        F.when(F.col("delay") < -60, "early")
         .when(F.col("delay") <= 120, "on_time")
         .otherwise("delayed"))
    .withColumn("is_delayed", F.col("delay")>120)
    .withColumn("is_stopped", F.col("speed")==0)
    .withColumn("is_moving", F.col("speed") > 0)
    .withColumn("gps_ok",    F.col("gpsQuality") > 0)
    .filter(F.col("lat").isNotNull() & F.col("lon").isNotNull())
)


In [0]:
routes_enrichment = (
    routes.select(
        F.col("route_id").alias("gtfs_route_id"),
        "route_type",
        "route_color",
        "route_text_color"
    )
    .dropDuplicates(["gtfs_route_id"])
)

In [0]:
vehicles_enrichment = (
    vehicles.select(
        "vehicleCode",
        "transportationType",
        "vehicleCharacteristics",
        "brand",
        "model",
        "productionYear",
        "length",
        "seats",
        "standingPlaces",
        "airConditioning",
        "wheelchairsRamp",
        "ticketMachine",
        "usb"
    )
    .dropDuplicates(["vehicleCode"])
)

In [0]:
silver_enriched = (
    silver.join(
        routes_enrichment, silver["routeId"] == routes_enrichment["gtfs_route_id"], "left"
    )
    .drop("gtfs_route_id")
    .join(
        vehicles_enrichment, on="vehicleCode",how="left")
)

In [0]:
silver_enriched.printSchema()
display(silver_enriched.limit(20))

In [0]:
silver = silver_enriched

In [0]:
(silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SCHEMA}.gps_positions_silver"))